# WhisperWatch live ranking compatibility

The follow-up compares archived and fresh Whisper Large V3 outputs on the original 80 feasibility clips. Use the **original, still-connected hosted GPU runtime**. In the pilot notebook, add two code cells at the bottom and copy the two code cells below into them. Keep the runtime connected so its recovered MP3s and installed GPU libraries stay available.

## 1. Upload the input bundle and compare

The bundle supplies the follow-up code, frozen boosted-tree model, and saved archived predictions used for verification. The run checks the recovered audio against the original SHA-256 values. It uses only the archived development metadata, which can be downloaded if its cache is missing. The same model scores both feature sets. Within each version of the same 80 clips, the existing blend averages the percentile ranks of boosted-tree risk and negative average Whisper log probability. Each queue reviews 16 clips, with ties broken by clip ID.

In [ ]:
from pathlib import Path
import sys
from zipfile import ZipFile
from google.colab import files

pilot_dir = Path("/content/whisperwatch_second_asr")
if not (pilot_dir / "audio").is_dir():
    raise RuntimeError("Open the connected pilot runtime that still has the recovered audio.")
print("Upload whisperwatch-compatibility-inputs.zip.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one ZIP file.")
bundle = Path(next(iter(uploaded)))
with ZipFile(bundle) as archive:
    (pilot_dir / "src/whisper_compatibility.py").write_bytes(archive.read("src/whisper_compatibility.py"))
sys.path.insert(0, str(pilot_dir))

from src.whisper_compatibility import run

output = pilot_dir / "compatibility_80"
summary = run(pilot_dir, bundle.resolve(), output)
from IPython.display import Markdown, display
display(Markdown((output / "report.md").read_text(encoding="utf-8")))


## 2. Download the results

The ZIP contains a short report, the complete comparison CSV, a feature drift table, both sets of segment records, and runtime settings. Error counts are specific to this deliberately selected batch. The original labels assess ranking changes on a fixed target. A separate comparison with the same text normalizer for both transcripts shows how many fresh-transcript errors each queue catches.

In [ ]:
from shutil import make_archive

download = make_archive("/content/whisperwatch-compatibility-results", "zip", root_dir=output)
files.download(download)
